# Week 9 Lab 1: Text Classification via Embeddings (Keras)

> **Goal**: Implement a complete NLP pipeline—Tokenization, Word Embeddings, and CNN processing—to classify movie reviews as Positive or Negative using TensorFlow/Keras. At the end, we export the model for Edge deployment.

## 1. Setup Data

**Concepts**: We use the standard IMDb database of 50,000 movie reviews. However, before using pre-tokenized IDs, we explore how to prepare **Raw Text** strings for a network.

### 1.1 Bonus: Processing Raw Strings

Computers cannot see "I love this movie!". We must map each word to an ID. Keras provides a layer that handles tokenization, lowercasing, and punctuation stripping in one go.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

# Sample raw data
raw_reviews = ["I love this movie!", "Absolute waste of time.", "Highly recommended strategy."]

# Create the vectorization layer
vectorize_layer = TextVectorization(max_tokens=1000, output_mode='int', output_sequence_length=10)

# Adapt the layer to the text (builds the vocabulary mapping list)
vectorize_layer.adapt(raw_reviews)

# Convert strings to integer tensors
integer_data = vectorize_layer(raw_reviews)
print(f"Integer Mappings:\n{integer_data.numpy()}")
print(f"Vocabulary: {vectorize_layer.get_vocabulary()}")

### 1.2 Loading the IMDb Benchmark

**Concepts**: Because text must be translated to numeric form, mapping dictionaries take massive amounts of RAM and disk space. Instead of raw text, Keras supplies the IMDb dataset already pre-tokenized into Integer IDs.

In [ ]:
from tensorflow.keras import layers, models, datasets
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load IMDb data. We restrict vocabulary to the top 10,000 most common words.
vocab_size = 10000
(train_data, train_labels), (test_data, test_labels) = datasets.imdb.load_data(num_words=vocab_size)

print(f"Training reviews: {len(train_data)}")
print(f"First review sample (encoded): {train_data[0][:10]}...")

## 2. Preprocessing (Padding Sequences)

**Task**: Reviews have variable lengths. A Neural Network expects fixed-length tensors inside a batch. We use `pad_sequences` to ensure every review is exactly 200 words long.

In [ ]:
maxlen = 200

train_padded = pad_sequences(train_data, maxlen=maxlen, padding='post', truncating='post')
test_padded = pad_sequences(test_data, maxlen=maxlen, padding='post', truncating='post')

print(f"Shape of padded training data: {train_padded.shape}")

## 3. Comparing Architectures

### 3.1 Baseline: Bag of Embeddings

This model averages all word vectors in a sentence. It ignores word order but is a strong benchmark.

In [ ]:
baseline_model = models.Sequential([
    layers.Embedding(vocab_size, 16, input_length=maxlen),
    layers.GlobalAveragePooling1D(),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

baseline_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
baseline_model.summary()

### 3.2 Advanced: The TextCNN Architecture

**Theory**: We start with an `Embedding` layer that projects our discrete word IDs into dense 64-dimensional vectors. 
Next, we slide a 1D Convolution over the text to look for patterns like 'not good' or 'masterpiece'. Finally, we use Global Max Pooling to squash the sequence down to a single vector.

In [ ]:
model = models.Sequential([
    layers.Embedding(input_dim=vocab_size, output_dim=64, input_length=maxlen),
    
    # Extract local N-gram features
    layers.Conv1D(128, 5, activation='relu'),
    layers.GlobalMaxPooling1D(),
    
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Binary Output (0 or 1)
])

model.summary()

## 4. Compile and Train

Since this is a binary classification (Positive/Negative), we must use `binary_crossentropy`.

In [ ]:
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_padded, train_labels, epochs=3, batch_size=64, 
                    validation_data=(test_padded, test_labels))